In [2]:
import sys
import os 

import numpy as np 
import pandas as pd 

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

sys.path.append(os.path.abspath("../.."))
from src.results_manager import ResultsManager

rm = ResultsManager("svm")

In [3]:
from src.load_processed import load_processed_data

dataset = load_processed_data("../../data/processed/preprocessed.npz")

X_train = dataset["X_train"]
X_test = dataset["X_test"]

y_train = dataset["y_train"]
y_test = dataset["y_test"]

font_test = dataset["font_test"]
italic_test = dataset["italic_test"]
strength_test = dataset["strength_test"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Number of classes:", len(np.unique(y_train)))
print("Data type:", X_train.dtype)

Train shape: (666136, 400)
Test shape: (166534, 400)
Number of classes: 256
Data type: float32


In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LinearSVC(
        C = 1.0,
        max_iter = 1000,
        tol = 1e-2,
        random_state = 42,
        verbose = 1,
        dual = False,
        multi_class = "ovr",
        loss = "squared_hinge"
    ))
])

Uputstvo: 
model mozete trenirati ispocetka (dug proces) ili koristiti vec istreniran i skladisten model. <br>Ukoliko zelite da trenirate model ponovo, izbrisite ga iz results/logistic_regression/model.pkl. <br>
Inace, samo pokrenite celiju, ona ce ucitati istrenirani model.

In [ ]:
import time 

trained_now = False 
training_time = None

model_loaded = rm.load_model()

if model_loaded is not None: 
    model = model_loaded
    print("Successfully loaded existing model.")
else:
    print("Training model ... ")
    start_time = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    print(f"Training done in {training_time:.2f} seconds")

    # save model using results manager 
    rm.save_model(model)

    trained_now = True 

In [ ]:

y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average = "macro")
weighted_f1 = f1_score(y_test, y_pred, average = "weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

In [ ]:
import json 

if training_time is None:
    training_time = -0.1

results = {
    "model": "svm_linear",
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "training_time_seconds": float(training_time)
}

# save results in .json file only if model is trained now: 
if trained_now:
    rm.save_metrics(results)


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(f"Shape: {cm.shape}\n")
print(cm)

In [ ]:
errors = []

n = cm.shape[0]

for true_class in range(n):
    for pred_class in range(n):
        if true_class != pred_class:            # avoid diag elements 
            count = cm[true_class, pred_class]
            errors.append((true_class, pred_class, count))

errors = sorted(errors, key = lambda x: x[2], reverse = True)

top_errors = errors[:10]

top_errors_readable = []
for true_class, pred_class, count in top_errors:
    label = f"{chr(true_class)} → {chr(pred_class)}"
    top_errors_readable.append((label, count))

errors_df = pd.DataFrame(
    top_errors_readable,
    columns = ["confusion", "count"]
)

rm.save_dataframe(errors_df, "top_confusions")

errors_df

### Per-class accuracy 

In [ ]:
# number of correct classifications per class (diag elements)
correct = np.diag(cm)

# total number of samples per class
total = cm.sum(axis = 1)

# per-class accuracy
per_class_accuracy = correct / total

# ASCII oznake klasa
labels = [f"{i}:{repr(chr(i))}" for i in range(cm.shape[0])]

# dataframe sa rezultatima
per_class_df = pd.DataFrame({
    "character": labels,
    "accuracy": per_class_accuracy,
    "correct": correct,
    "total": total,
    "percentage_%": per_class_accuracy *100
})

# sortiranje po tacnosti
per_class_df = per_class_df.sort_values("accuracy")
rm.save_dataframe(per_class_df, "class_accuracy")

worst_classified = per_class_df.head(10)
rm.save_dataframe(worst_classified, "worst_classified_classes")
print("Worst classified characters:")
print(worst_classified)

best_classified = per_class_df.tail(10)
rm.save_dataframe(best_classified, "best_classified_classes")
print("\nBest classified characters:")
print(best_classified)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

font_results = []

for font in np.unique(font_test):
    mask = font_test == font

    acc = accuracy_score(
        y_test[mask],
        y_pred[mask]
    )

    font_results.append({
        "font": font,
        "samples": mask.sum(),
        "accuracy": acc
    })


font_df = pd.DataFrame(font_results)
font_df = font_df.sort_values("accuracy", ascending = False)
font_df["accuracy_%"] = (font_df["accuracy"] * 100).round(2)
rm.save_dataframe(font_df, "font_accuracy")

font_df

In [ ]:
italic_results = []

for value in np.unique(italic_test):
    mask = italic_test == value 

    acc = accuracy_score(
        y_test[mask],
        y_pred[mask]
    )

    italic_results.append({
        "italic": value, 
        "samples": mask.sum(),
        "accuracy": acc
    })

italic_df = pd.DataFrame(italic_results)
italic_df["accuracy_%"] = (italic_df["accuracy"] * 100).round(2)
rm.save_dataframe(italic_df, "italic_accuracy")

italic_df

57.13% karaktera koji nisu italic je klasifikovano tacno  
26.15% karaktera koji jesu italic je klasifikovano tacno

In [ ]:
bins = np.linspace(0, 1, 6)   # 0.0, 0.2, 0.4, 0.6, 0.8, 1.0
labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]

strength_results = []

for i in range(len(bins)-1):

    mask = (strength_test >= bins[i]) & (strength_test < bins[i+1])

    samples = mask.sum()
    if samples == 0:
        acc = np.nan
    else: 
        acc = accuracy_score(
            y_test[mask],
            y_pred[mask]
        )


    strength_results.append({
        "strength_range": labels[i],
        "samples": samples,
        "accuracy": acc
    })


strength_df = pd.DataFrame(strength_results)
strength_df["accuracy_%"] = (strength_df["accuracy"] * 100).round(2)

rm.save_dataframe(strength_df, "strength_accuracy")

strength_df

Deblji karakteri imaju više spojenih piksela, pa se oblici slova međusobno približavaju.  
Na primer: 8 vs B, 5 vs S, O vs 0 -> jos slicniji kada je font deblji